Aims-vcr


In [1]:
!pip install -q git+https://github.com/huggingface/transformers accelerate qwen-vl-utils bitsandbytes peft trl

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.3 MB/s eta 0:00:00


In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [5]:
from google.colab import userdata
from huggingface_hub import login
import wandb

login(token=userdata.get("HF_TOKEN"))
wandb.login(key=userdata.get("WANDB_API_KEY"))

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tanygupt360 (tanygupt360-delhi-technological-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

model_name = "Qwen/Qwen3-VL-4B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
)
processor = AutoProcessor.from_pretrained(model_name)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.067, #SIX SEVENN
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/64.7k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

[ERROR] `min_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.12/dist-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.
[ERROR] `max_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.12/dist-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.
trainable params: 16,515,072 || all params: 4,454,330,880 || trainable%: 0.3708


In [8]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3VLForConditionalGeneration(
      (model): Qwen3VLModel(
        (visual): Qwen3VLVisionModel(
          (patch_embed): Qwen3VLVisionPatchEmbed(
            (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
          )
          (pos_embed): Embedding(2304, 1024)
          (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-23): 24 x Qwen3VLVisionBlock(
              (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
              (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
              (attn): Qwen3VLVisionAttention(
                (qkv): Linear4bit(in_features=1024, out_features=3072, bias=True)
                (proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
              )
              (mlp): Qwen3VLVisionMLP(
                (linear_fc1): Linear4bit(in_features=1024, out_features=4096, bias=True)
   

In [9]:
from PIL import Image, ImageDraw
import requests
from io import BytesIO
from trl import SFTConfig, SFTTrainer

In [10]:
correct_answers = 0
correct_rationales = 0
output = []
predicted_answer = ""

In [11]:
s_num = 1

In [32]:
from datasets import load_dataset

train_question_db = load_dataset(
    "Rowan/vcr",
    name="questions",
    split="train[200:400]"
)

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

In [33]:
needed_images = set(train_question_db["img_fn"])
print(len(needed_images))

image_ds = load_dataset(
    "Rowan/vcr",
    name="image_examples",
    split="train",
    streaming = True
)

image_cache = {}
for img_sample in image_ds:
    fn = img_sample["img_fn"]
    if fn in needed_images:
        image_cache[fn] = img_sample
    if len(image_cache) == len(needed_images):
        break

69


Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

In [34]:
def preprocess_image(image_record):

    image = image_record["image"].convert("RGB")

    orig_w, orig_h = image.size

    image.thumbnail((768, 768))

    new_w, new_h = image.size

    scale_x = new_w / orig_w
    scale_y = new_h / orig_h

    boxes = []

    for box in image_record["boxes"]:
      x1, y1, x2, y2, confidence = box

      boxes.append([
          x1 * scale_x,
          y1 * scale_y,
          x2 * scale_x,
          y2 * scale_y,
          confidence
      ])

    draw = ImageDraw.Draw(image)

    for i, box in enumerate(boxes):

      x1, y1, x2, y2, confidence = box

      draw.rectangle(
          [x1, y1, x2, y2],
          outline="red",
          width=2
      )

      draw.text(
          ((x1+x2)/2, (y1+y2)/2),
          f"{image_record['objects'][i]}{i}",
          fill="green",
          font_size = 18
      )
    # display(image)
    return image

In [35]:
def build_example(sample, image_record):
  image = preprocess_image(image_record)

  correct_idx = sample["answer_label"]
  answer_letter = "ABCD"[correct_idx]

  return {
    "images": [image],
    "messages": [
      {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": aPrompt(sample)},
      ]},
      {"role": "assistant", "content": [
        {"type": "text", "text": answer_letter},
      ]},
    ]
  }

def build_rationale_example(sample, image_record):
  image = preprocess_image(image_record)

  correct_answer_idx = sample["answer_label"]
  correct_answer_text = sample["answer_choice_texts"][correct_answer_idx]
  correct_rationale_letter = "ABCD"[sample["rationale_label"]]

  # Build the rationale prompt using the KNOWN correct answer (not a model prediction)
  question = sample["question_text"]
  rationales = sample["rationale_choice_texts"]
  rationale_prompt = f"""
You are solving the rationale-selection stage of a Visual Commonsense Reasoning (VCR) task.

The model has already selected the following answer:

Selected answer:
{correct_answer_text}

Your task is to identify the ONE rationale that best explains why that selected answer is correct.

Rationale choices:

A. {rationales[0]}
B. {rationales[1]}
C. {rationales[2]}
D. {rationales[3]}

Respond with ONLY the letter of the correct rationale: A, B, C, or D.
"""

  return {
    "images": [image],
    "messages": [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": rationale_prompt},
        ]},
        {"role": "assistant", "content": [
          {"type": "text", "text": correct_rationale_letter},
        ]},
      ]
  }

In [36]:
train_question_db

Dataset({
    features: ['movie', 'objects', 'interesting_scores', 'answer_likelihood', 'img_fn', 'metadata_fn', 'answer_orig', 'question_orig', 'rationale_orig', 'question_tokens', 'answer_choice_tokens', 'answer_label', 'answer_match_iter', 'answer_sources', 'rationale_choice_tokens', 'rationale_sources', 'rationale_match_iter', 'rationale_label', 'source_zip_crc_mismatch', 'img_id', 'question_number', 'annot_id', 'match_fold', 'match_index', 'question_text', 'answer_choice_texts', 'rationale_choice_texts'],
    num_rows: 200
})

In [37]:
formatted = []
for s in train_question_db:
    if s["img_fn"] not in image_cache:
        continue
    formatted.append(build_example(s, image_cache[s["img_fn"]]))            # answer task
    formatted.append(build_rationale_example(s, image_cache[s["img_fn"]]))  # rationale task

train_dataset = Dataset.from_list(formatted)

In [25]:
# def build_rationale_example(sample, image_record):
#     image = preprocess_image(image_record)

#     correct_answer_idx = sample["answer_label"]
#     correct_answer_text = sample["answer_choice_texts"][correct_answer_idx]
#     correct_rationale_letter = "ABCD"[sample["rationale_label"]]

#     question = sample["question_text"]
#     rationales = sample["rationale_choice_texts"]
#     rationale_prompt = f"""
# You are solving the rationale-selection stage of a Visual Commonsense Reasoning (VCR) task.

# The model has already selected the following answer:

# Selected answer:
# {correct_answer_text}

# Your task is to identify the ONE rationale that best explains why that selected answer is correct.

# Rationale choices:

# A. {rationales[0]}
# B. {rationales[1]}
# C. {rationales[2]}
# D. {rationales[3]}

# Respond with ONLY the letter of the correct rationale: A, B, C, or D.
# """

#     return {
#         "images": [image],
#         "messages": [
#             {"role": "user", "content": [
#                 {"type": "image"},
#                 {"type": "text", "text": rationale_prompt},
#             ]},
#             {"role": "assistant", "content": [
#                 {"type": "text", "text": correct_rationale_letter},
#             ]},
#         ]
#     }

# rationale_formatted = [
#     build_rationale_example(s, image_cache[s["img_fn"]])
#     for s in train_question_db
#     if s["img_fn"] in image_cache
# ]
# rationale_dataset = Dataset.from_list(rationale_formatted)

In [17]:
result = preprocess_image(image_cache[train_question_db[0]["img_fn"]])
print(type(result))

<class 'PIL.Image.Image'>


In [38]:
model.train()
model.gradient_checkpointing_enable()
model.config.use_cache = False

from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
  output_dir="./qwen3vl-vcr-lora-v2",
  per_device_train_batch_size=1,
  gradient_accumulation_steps=1,
  max_steps=20,                      # number of stuff i need
  learning_rate=2e-4,
  bf16=True,
  fp16=False,
  optim="adamw_torch",
  gradient_checkpointing=True,
  logging_steps=5,
  save_strategy="steps",
  save_steps=20,                     # save at the end, yummerz
  remove_unused_columns=False,

  # Hugging Face Hub
  push_to_hub=True,
  hub_model_id="zzephyrr/qwen3vl-vcr-lora-v2",
  # W&B
  report_to="wandb",
  run_name="qwen3vl-4b-vcr-lora-t4-run2",
)

trainer = SFTTrainer(
  model=model,
  args=training_args,
  train_dataset=train_dataset,
  # train_dataset=rationale_dataset,
  processing_class=processor,
)
trainer.train()
trainer.push_to_hub()   # pushes the LoRA adapter (and model card) to the HFHub

Step,Training Loss
5,3.763125
10,4.413801
15,3.892903
20,4.488194


No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/zzephyrr/qwen3vl-vcr-lora-v2/commit/a91ac537cea8b12d5b8c636cccb6e637c4c8f9d1', commit_message='End of training', commit_description='', oid='a91ac537cea8b12d5b8c636cccb6e637c4c8f9d1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/zzephyrr/qwen3vl-vcr-lora-v2', endpoint='https://huggingface.co', repo_type='model', repo_id='zzephyrr/qwen3vl-vcr-lora-v2'), pr_revision=None, pr_num=None)

In [6]:
def aPrompt(sample):

  question = sample["question_text"]
  answers = sample["answer_choice_texts"]

  prompt = f"""
You are solving a Visual Commonsense Reasoning (VCR) multiple-choice question.

Your task is to identify the ONE answer choice that is best supported by the image and the question.

Follow this reasoning process internally:

1. Carefully inspect the image.
2. Identify the people, objects, actions, relationships, and relevant visual details.
3. Determine exactly what the question is asking.
4. Evaluate each answer choice against the visual evidence and the question.
5. Reject choices that contradict the image, answer a different question, or rely on unsupported assumptions.
6. Select the choice with the strongest evidence.

Important:
- Base your decision primarily on evidence visible in the image.
- Use ordinary commonsense only when it is necessary to interpret the situation.
- Do not choose an answer merely because it sounds plausible.
- Pay close attention to the distinction between similar people, objects, actions, and relationships.
- There is exactly ONE correct answer.

Question:
{question}

Answer choices:

A. {answers[0]}
B. {answers[1]}
C. {answers[2]}
D. {answers[3]}

After reasoning internally, respond with ONLY the letter of the correct choice:
A, B, C, or D.
"""
  return prompt
  # print(prompt)

In [20]:
def rPrompt(sample):
  global output
  global predicted_answer

  predicted_answer = output[0].strip()
  question = sample["question_text"]
  rationales = sample["rationale_choice_texts"]

  rationale_prompt = f"""
You are solving the rationale-selection stage of a Visual Commonsense Reasoning (VCR) task.

The model has already selected the following answer:

Selected answer:
{predicted_answer}

Your task is to identify the ONE rationale that best explains why that selected answer is correct.

Reason internally through the following steps:

1. Re-examine the image.
2. Consider the question and the selected answer together.
3. Determine what visual evidence or commonsense connection supports the selected answer.
4. Compare every rationale choice.
5. Reject rationales that:
   - contradict the image,
   - do not explain the selected answer,
   - refer to the wrong person/object/action,
   - introduce unsupported information,
   - or are logically weaker than another choice.
6. Select the rationale that provides the strongest explanation for the selected answer.

Important:
- The rationale must explain the SELECTED ANSWER, not merely describe the image.
- Do not choose a rationale just because it is generally plausible.
- Pay attention to people, actions, objects, and relationships mentioned in the question.
- There is exactly ONE correct rationale.

Question:
{question}

Selected answer:
{predicted_answer}

Rationale choices:

A. {rationales[0]}
B. {rationales[1]}
C. {rationales[2]}
D. {rationales[3]}

After reasoning internally, respond with ONLY the letter of the best rationale:
A, B, C, or D.
"""
  return rationale_prompt
  # print(rationale_prompt)

In [21]:
def output_generator(sample, image, prompt, answer_or_rationale):
  global output
  global predicted_answer
  global correct_answers
  global correct_rationales
  global s_num

  messages = [
      {"role": "user", "content": [
          {"type": "image", "image": image},
          {"type": "text", "text": prompt},
      ]}
  ]

  text = processor.apply_chat_template(
      messages, tokenize=False, add_generation_prompt=True
  )

  inputs = processor(
      text=[text], images=[image], padding=True, return_tensors="pt",
  ).to(model.device)

  with torch.no_grad():
      generated_ids = model.generate(
          **inputs,
          max_new_tokens=1,
          do_sample=False,
      )

  input_length = inputs["input_ids"].shape[1]
  generated_ids = generated_ids[:, input_length:]

  output = processor.batch_decode(generated_ids, skip_special_tokens=True)

  if answer_or_rationale:
    correct = "ABCD"[sample["answer_label"]]
    is_correct = output[0].strip() == correct
    print(f"[{s_num}] Answer  -> Model: {output[0].strip()!r}  Correct: {correct}  {'✅' if is_correct else '❌'}")
    s_num += 1
    if is_correct:
      correct_answers += 1
  else:
    correct = "ABCD"[sample["rationale_label"]]
    is_correct = output[0].strip() == correct
    print(f"     Rationale -> Model: {output[0].strip()!r}  Correct: {correct}  {'✅' if is_correct else '❌'}")
    if is_correct:
      correct_rationales += 1

In [39]:
model.eval()
model.gradient_checkpointing_disable()
model.config.use_cache = True

In [40]:
from datasets import load_dataset

eval_question_db = load_dataset("Rowan/vcr", name="questions", split="train[200:230]")  # next 30, no overlap

eval_needed_images = set(eval_question_db["img_fn"])
eval_image_ds = load_dataset("Rowan/vcr", name="image_examples", split="train", streaming=True)

eval_image_cache = {}
for img_sample in eval_image_ds:
    if img_sample["img_fn"] in eval_needed_images:
        eval_image_cache[img_sample["img_fn"]] = img_sample
    if len(eval_image_cache) == len(eval_needed_images):
        break

samples = list(eval_question_db)
image_cache = eval_image_cache   # output_generator/loop below reads from this name

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

In [ ]:
correct_answers = 0
correct_rationales = 0
s_num = 1

for sample in samples:
  image_record = image_cache[sample["img_fn"]]
  image = preprocess_image(image_record)

  prompt_a = aPrompt(sample)
  output_generator(sample, image, prompt_a, True)
  prompt_r = rPrompt(sample)
  output_generator(sample, image, prompt_r, False)

n = len(samples)
print(f"\nAnswer accuracy:    {correct_answers}/{n} ({100*correct_answers/n:.1f}%)")
print(f"Rationale accuracy: {correct_rationales}/{n} ({100*correct_rationales/n:.1f}%)")

In [ ]:
# for sample in samples:

#   image_record = image_cache[sample["img_fn"]]

#   image = preprocess_image(image_record)

#   prompt_a = aPrompt(sample)
#   output_generator(sample,image,prompt_a,True)
#   prompt_r = rPrompt(sample)
#   output_generator(sample,image,prompt_r,False)

# # print(correct_answers)
# # print(correct_rationales)


In [ ]:
# print(correct_answers)
# print(correct_rationales)